# Exercise 2: BLE Localization
Pietro Pizzoccheri 10797420 [team leader]  
Lorenzo Bardelli 10831941

## Questions:

Design an improved IoT solution.
Your answer should include:
1. The logic used to for BLE beaconing;
2. Any additional sensors added to the system;
3. Pseudocode of the improved firmware;
4. One or more real commercial components that could be used to implement the
proposed sensing logic;
5. An estimate of the expected energy saving with respect to the original periodic
beaconing solution.
In the energy-saving estimate, clearly state all assumptions including the average
time per day in which the denture is expected to require localization

## Baseline firmware
```text
loop forever:
    send_BLE_beacon()
    sleep(10 seconds)
```
This sends BLE advertisements all day, even when localization is not useful.


## Improved solution
The denture remains a pure BLE beacon:
- denture = BLE broadcaster
- gateways = BLE observers collecting RSSI for localization

improved policy:
```text
if denture is in mouth:
    send rare heartbeat advertisements

if denture is in normal storage:
    send rare heartbeat advertisements

otherwise:
    send advertisements every 10 seconds for localization
```


## Feasibility and component choices

### Route A (case-based storage)
If the nursing home uses a dedicated denture case, we would use a temperature sensor plus a Hall-effect sensor. The temperature sensor tells whether the denture is probably in the patient’s mouth, while the Hall sensor detects whether it is inside the case by sensing a small magnet embedded in the case

Use:
- Temperature sensor: TI TMP117
- Case detector: TI DRV5032 Hall-effect switch

Logic:
- high temperature -> likely in mouth
- Hall active -> in case
- low temperature + Hall inactive -> likely misplaced -> advertise every 10 s

We would place the electronics in a small sealed pocket in the thickest part of the denture base. The temperature sensor should stay close to the external surface so it can react to mouth temperature, while the Hall sensor should be placed on the side facing the case magnet

The advantage is that this solution is simple and reliable. The possible issue is that it only works if patients actually use the dedicated magnetic case, but with caregivers surveillance it can be effectively monitored

### Route B (water-based storage)
If patients usually store the denture in water at night, we would use a humidity and temperature sensor instead. High temperature means the denture is probably in the mouth. Low temperature with high humidity means it is probably stored in water. Low temperature with low humidity means it may be misplaced, so the BLE beacon should advertise more frequently

Use:
- Humidity + temperature sensor: TI HDC3022

Logic:
- high temperature -> likely in mouth
- low temperature + high humidity -> likely stored in water
- low temperature + low humidity -> likely misplaced -> advertise every 10 s

We would place the humidity/temperature sensor close to the outer surface of the denture, because it needs to sense the surrounding humidity. The BLE chip and battery should remain sealed deeper inside the prosthesis

The advantage is that this route does not require a special case. The possible issue is that humidity sensing is more delicate, because the sensor must be exposed enough to detect water/humidity but still protected from damage


In [4]:
# energy-saving estimate (same assumptions for both routes)
# original firmware: one advertisement every 10 seconds all day
N_original = 24 * 3600 / 10

# assumptions
hours_worn = 12
hours_stored = 11
hours_localization = 1

heartbeat_interval = 30 * 60   # 30 minutes
active_interval = 10           # 10 seconds

N_worn = hours_worn * 3600 / heartbeat_interval
N_stored = hours_stored * 3600 / heartbeat_interval
N_localization = hours_localization * 3600 / active_interval

N_improved = N_worn + N_stored + N_localization
saving = 1 - N_improved / N_original

print(f'Original advertisements/day = {N_original:.0f}')
print(f'Improved advertisements/day = {N_improved:.0f}')
print(f'Saving = {saving*100:.1f}%')

# route A and route B have same count under these assumptions
print('\nRoute A (case-based):  ', f'{N_improved:.0f} adv/day')
print('Route B (water-based):', f'{N_improved:.0f} adv/day')


Original advertisements/day = 8640
Improved advertisements/day = 406
Saving = 95.3%

Route A (case-based):   406 adv/day
Route B (water-based): 406 adv/day


## Firmware pseudocode
### Route A firmware (case-based: temperature + Hall)
```text
constants:
    HEARTBEAT_INTERVAL = 30 minutes
    ACTIVE_ADV_INTERVAL = 10 seconds
    MOUTH_TEMP_MIN = 32 C

setup:
    init BLE advertising
    init temperature sensor
    init Hall sensor

loop forever:
    temperature = read_temperature()
    hall_active = read_Hall_sensor()

    in_mouth = (temperature >= MOUTH_TEMP_MIN)
    in_case = hall_active

    if in_mouth or in_case:
        state = SAFE
    else:
        state = LOCALIZATION_NEEDED

    send advertisement with (denture_id, battery_level, state)

    if state == LOCALIZATION_NEEDED:
        sleep(ACTIVE_ADV_INTERVAL)
    else:
        sleep(HEARTBEAT_INTERVAL)
```

### Route B firmware (water-based: temperature + humidity)
```text
constants:
    HEARTBEAT_INTERVAL = 30 minutes
    ACTIVE_ADV_INTERVAL = 10 seconds
    MOUTH_TEMP_MIN = 32 C
    STORAGE_HUMIDITY_MIN = 80 %

setup:
    init BLE advertising
    init temperature sensor
    init humidity sensor

loop forever:
    temperature = read_temperature()
    humidity = read_humidity()

    in_mouth = (temperature >= MOUTH_TEMP_MIN)
    in_water_storage = (humidity >= STORAGE_HUMIDITY_MIN)

    if in_mouth or in_water_storage:
        state = SAFE
    else:
        state = LOCALIZATION_NEEDED

    send advertisement with (denture_id, battery_level, state)

    if state == LOCALIZATION_NEEDED:
        sleep(ACTIVE_ADV_INTERVAL)
    else:
        sleep(HEARTBEAT_INTERVAL)
```


## Final answer
The denture stays a pure BLE broadcaster and only changes advertisement interval using local context:
- in mouth or correctly stored -> rare heartbeat
- otherwise -> advertise every 10 seconds for localization

With the assumptions explained above, the advertisement count goes from 8640/day to 406/day, which is about 95.3 percent reduction